In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 10-K Text Mining Process
## 1. Import the required Python libraries.
## 2. Load the original 10-K TXT files
## 3. Remove HTML tags using BeautifulSoup.
## 4. Save the cleaned TXT files.
## 5. Load the Loughran–McDonald (LM) Dictionary.
## 6. Extract positive, negative, and litigious word lists.
## 7. Load the 100 cleaned 10-K TXT files.
## 8. Count positive, negative, and litigious word occurrences and Calculate positivity, negativity, and litigious rates.
## 9. Create a Pandas DataFrame with 100 rows and 8 columns.
## 10. Export the final results to a CSV file.

# Import the libraries

In [ ]:
import os

# re (Regular Expressions): Used to search, extract, and clean text based on specific patterns.
import re

#glob: Used to find files that match a specific pattern, such as all .txt files in a folder.
import glob

import pandas as pd
from bs4 import BeautifulSoup


# Load the original 10-k TXT files

In [ ]:
folder = "YOUR_DATA_DIRECTORY"

files = os.listdir(folder)

for file in files:
    print(file)

In [ ]:
input_folder = "YOUR_DATA_DIRECTORY"

output_folder = os.path.join(input_folder, "Cleaned_10K")

os.makedirs(output_folder, exist_ok=True)

files = glob.glob(os.path.join(input_folder, "*.txt"))
print("Total files:", len(files))

# Remove HTML tags using BeautifulSoup.

In [ ]:
for file in files:

    with open(file, "r", encoding="utf-8", errors="replace") as f:
        html = f.read()

    #cleaned = BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)

    cleaned = BeautifulSoup(html, "lxml").get_text(separator=" ", strip=True)

    # Cleaned_10K save
    output_path = os.path.join(output_folder, os.path.basename(file))

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned)

    print("Completed:", output_path)

# Load the LM Dictionary and check the data set

In [ ]:
import pandas as pd

#Counter: Used to count how many times each word appears in a text.
from collections import Counter

LM_path = "YOUR_DATA_DIRECTORY"

# Load the LM Dictionary
LM = pd.read_csv(LM_path)

# Check the dataset
LM.columns

LM.head(20)

# Create Positive, Negative, and Litigious Word Lists

In [ ]:
results = []

#From panda.pydata.org
#Boolean indexing: Select rows where df.A is greater than 0. df[df["A"] > 0]
#Selection by label: Selecting a row matching a label df.loc[dates[0]]
#String Methods: converting words to uppercase or lowercase s.str.lower()


# Since the words in the LM Dictionary are generally provided in uppercase, we standardized our text to uppercase.
# LM["Positive"] retrieves the actual numerical data from the Positive column, while "Positive" is simply a string representing the column name.

# Positive words
positive_words = set(
    LM.loc[LM["Positive"] > 0, "Word"].str.upper()
)

# Negative words
negative_words = set(
    LM.loc[LM["Negative"] > 0, "Word"].str.upper()
)

# Litigious words
litigious_words = set(
    LM.loc[LM["Litigious"] > 0, "Word"].str.upper()
)

# Load the cleaned files

In [ ]:
# Find all cleaned TXT files
cleaned_files = sorted(
    glob.glob(os.path.join(output_folder, "*.txt"))
)

print("Total files:", len(cleaned_files))

# Perform Sentiment Analysis on the 100 TXT Files

In [ ]:
# Create an empty list to store the results
results = []

# Analyze each cleaned TXT file
for file in cleaned_files:

    # 1. Read the TXT file
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    # 2. Tokenize the text
    words = re.findall(r"[A-Z]+", text.upper())

    # 3. Count the total number of words
    wordCount = len(words)

    # 4. Calculate the frequency of each word
    word_counts = Counter(words)

    # 5. Count positive words
    PositiveWordCount = sum(
        word_counts[word] for word in positive_words
    )

    # 6. Count negative words
    NegativeWordCount = sum(
        word_counts[word] for word in negative_words
    )

    # 7. Count litigious words
    LitigiousWordCount = sum(
        word_counts[word] for word in litigious_words
    )

    # 8. Calculate sentiment rates
    PositivityRate = PositiveWordCount / wordCount if wordCount else 0

    NegativityRate = NegativeWordCount / wordCount if wordCount else 0

    LitigiousRate = LitigiousWordCount / wordCount if wordCount else 0

    # 9. Store the results
    results.append({
        "filename": os.path.basename(file),
        "wordCount": wordCount,
        "PositiveWordCount": PositiveWordCount,
        "NegativeWordCount": NegativeWordCount,
        "LitigiousWordCount": LitigiousWordCount,
        "PositivityRate": PositivityRate,
        "NegativityRate": NegativityRate,
        "LitigiousRate": LitigiousRate
    })

    print("Completed:", os.path.basename(file))

# Convert the Results into a DataFrame

In [ ]:
df = pd.DataFrame(results)

print(df.shape)

df.head()

# Save the Results as a csv File

In [ ]:
# Define the output file path
result_path = os.path.join(input_folder, "10K_Sentiment_Results.csv")

# Save the results
df.to_csv(result_path, index=False, encoding="utf-8-sig")

print("Saved:", result_path)

In [ ]:
print("Rows:", len(df))


df.head(100)